# Installing dependencies

In [1]:
import pandas as pd
import numpy as np

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Axel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

1. Load Dataset

In [3]:
df = pd.read_csv('IMDB Dataset.csv')

print(df.head(10))
print(df.info())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
5  Probably my all-time favorite movie, a story o...  positive
6  I sure would like to see a resurrection of a u...  positive
7  This show was an amazing, fresh & innovative i...  negative
8  Encouraged by the positive comments about this...  negative
9  If you like original gut wrenching laughter yo...  positive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
N

2. Data cleaning

In [4]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)  # remove symbols
    words = text.split()
    
    words = [ps.stem(word) for word in words if word not in stop_words]
    
    return ' '.join(words)

df['clean_review'] = df['review'].apply(clean_text)

3. Label encoding

In [5]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

4. Train & test split

In [6]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

5. TF-IDF

In [7]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

6. Naive Bayes model

In [8]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

7. Evaluation

In [9]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8517

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.85      0.85      4961
           1       0.85      0.86      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


Confusion Matrix:
 [[4193  768]
 [ 715 4324]]


8. Test custom

In [10]:
def predict_sentiment(text):
    text_clean = clean_text(text)
    vector = vectorizer.transform([text_clean]).toarray()
    prediction = model.predict(vector)[0]
    
    return "Positive 😊" if prediction == 1 else "Negative 😡"

# Example
print(predict_sentiment("This movie was amazing!"))
print(predict_sentiment("Worst film ever"))

Positive 😊
Negative 😡


9. Save model

In [11]:
import pickle

pickle.dump(model, open('naive_bayes_model.pkl', 'wb'))
pickle.dump(vectorizer, open('tfidf_vectorizer.pkl', 'wb'))